# 🌏 SE Asia Tectonic Evolution Animation

This notebook creates a high-quality animation of Southeast Asia's tectonic evolution from **240 million years ago (Ma)** to the present day.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/seasia-tectonic-animation/blob/main/seasia_tectani_gplately.ipynb)

## Features
- 🗺️ Reconstructed coastlines through geological time
- 🔴 Subduction zones (convergent boundaries)
- 🔵 Mid-ocean ridges (divergent boundaries)
- 🟠 Transform faults (strike-slip boundaries)

**Author:** Tin Ko Oo, Mahidol University, Thailand

## 1. Install Dependencies

First, let's install the required packages. This may take a few minutes.

In [ ]:
# Install required packages
!pip install gplately cartopy -q

# Install ffmpeg for video encoding
!apt-get install ffmpeg -qq

print("✅ Installation complete!")

## 2. Import Libraries

In [ ]:
import gplately
from gplately import PlateReconstruction, PlotTopologies
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np
from matplotlib.animation import FuncAnimation
from matplotlib.lines import Line2D
from IPython.display import HTML, Video
import warnings
warnings.filterwarnings("ignore")

print(f"✅ GPlately version: {gplately.__version__}")

## 3. Download Plate Reconstruction Data

GPlately automatically downloads and caches the plate model files from EarthByte servers.

**Available models:**
- `Muller2019` - Global reconstructions 0-240 Ma (recommended)
- `Muller2022` - Deep time reconstructions 0-1000 Ma
- `Merdith2021` - Supercontinent cycles 0-1000 Ma

In [ ]:
print("📥 Downloading plate reconstruction data...")
print("   (This may take 1-2 minutes on first run)\n")

# Choose plate model
MODEL_NAME = "Muller2019"

# Initialize data server
gdownload = gplately.DataServer(MODEL_NAME)

# Download plate reconstruction files
rotation_model, topology_features, static_polygons = gdownload.get_plate_reconstruction_files()

# Download geometry files
coastlines, continents, COBs = gdownload.get_topology_geometries()

# Create the plate reconstruction model
model = PlateReconstruction(rotation_model, topology_features, static_polygons)

print("✅ Data download complete!")

## 4. Configuration

Adjust the animation parameters below:

In [ ]:
# ============================================
# ANIMATION SETTINGS - Modify these as needed
# ============================================

# Time range (millions of years ago)
TIME_START = 240        # Start time (Ma) - Late Triassic
TIME_END = 0            # End time (Ma) - Present day
TIME_STEP = 2           # Time step per frame (Ma)

# Video settings
FPS = 15                # Frames per second
DPI = 150               # Resolution (higher = better quality but larger file)
OUTPUT_FILE = 'SE_Asia_tectonic_evolution_240Ma.mp4'

# Geographic extent [lon_min, lon_max, lat_min, lat_max]
# SE Asia region
EXTENT = [90, 145, -15, 30]

# Alternative extents:
# EXTENT = [-180, 180, -90, 90]    # Global view
# EXTENT = [95, 140, -12, 8]       # Indonesia focus
# EXTENT = [95, 115, 5, 25]        # Thailand/Indochina

# Calculate number of frames
n_frames = int((TIME_START - TIME_END) / TIME_STEP) + 1
duration = n_frames / FPS

print(f"📊 Animation Configuration:")
print(f"   Time range: {TIME_START} Ma → {TIME_END} Ma")
print(f"   Time step: {TIME_STEP} Ma")
print(f"   Total frames: {n_frames}")
print(f"   Duration: {duration:.1f} seconds")
print(f"   Output: {OUTPUT_FILE}")

## 5. Preview a Single Time Slice

Let's preview what the reconstruction looks like at a specific time:

In [ ]:
# Preview time (Ma)
preview_time = 100  # Change this to preview different times

# Create figure
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection=ccrs.PlateCarree())
ax.set_extent(EXTENT, crs=ccrs.PlateCarree())

# Add gridlines
gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5)
gl.top_labels = False
gl.right_labels = False

# Create PlotTopologies object
gplot = PlotTopologies(model, coastlines=coastlines, continents=continents, COBs=COBs, time=preview_time)

# Plot features
try:
    gplot.plot_continents(ax, facecolor='lightgray', edgecolor='none', alpha=0.5)
except: pass

try:
    gplot.plot_coastlines(ax, color='black', linewidth=1.2)
except: pass

try:
    gplot.plot_subduction_teeth(ax, color='red', linewidth=1.5)
except: pass

try:
    gplot.plot_ridges(ax, color='blue', linewidth=1.5)
except: pass

try:
    gplot.plot_transforms(ax, color='orange', linewidth=1.2)
except: pass

# Add title and legend
ax.set_title(f'SE Asia Tectonic Reconstruction: {preview_time} Ma', fontsize=14)

legend_elements = [
    Line2D([0], [0], color='black', linewidth=1.5, label='Coastlines'),
    Line2D([0], [0], color='red', linewidth=2, label='Subduction Zones'),
    Line2D([0], [0], color='blue', linewidth=2, label='Mid-Ocean Ridges'),
    Line2D([0], [0], color='orange', linewidth=1.5, label='Transform Faults'),
]
ax.legend(handles=legend_elements, loc='lower left', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\n🗺️ Preview of {preview_time} Ma reconstruction")

## 6. Create the Animation

⚠️ **Note:** This step may take 10-30 minutes depending on your settings.

In [ ]:
print(f"🎬 Creating animation with {n_frames} frames...")
print(f"   Estimated time: {n_frames * 2} - {n_frames * 5} seconds\n")

# Set up the figure
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection=ccrs.PlateCarree())

# Create PlotTopologies object
gplot = PlotTopologies(model, coastlines=coastlines, continents=continents, COBs=COBs, time=0)

def update(frame):
    reconstruction_time = TIME_START - frame * TIME_STEP
    if reconstruction_time < TIME_END:
        reconstruction_time = TIME_END
    
    # Clear and reset axes
    ax.clear()
    ax.set_extent(EXTENT, crs=ccrs.PlateCarree())
    
    # Add gridlines
    gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5)
    gl.top_labels = False
    gl.right_labels = False
    
    # Update time
    gplot.time = reconstruction_time
    
    # Plot all features
    try:
        gplot.plot_continents(ax, facecolor='lightgray', edgecolor='none', alpha=0.5)
    except: pass
    
    try:
        gplot.plot_coastlines(ax, color='black', linewidth=1.2)
    except: pass
    
    try:
        gplot.plot_subduction_teeth(ax, color='red', linewidth=1.5)
    except: pass
    
    try:
        gplot.plot_ridges(ax, color='blue', linewidth=1.5)
    except: pass
    
    try:
        gplot.plot_transforms(ax, color='orange', linewidth=1.2)
    except: pass
    
    try:
        gplot.plot_trenches(ax, color='darkred', linewidth=1.5)
    except: pass
    
    # Add title and time label
    ax.set_title('SE Asia Tectonic Evolution', fontsize=14)
    ax.text(0.02, 0.98, f'{reconstruction_time:.0f} Ma', 
            transform=ax.transAxes, fontsize=16, va='top', ha='left',
            bbox=dict(boxstyle="round", facecolor='wheat'))
    
    # Add legend
    legend_elements = [
        Line2D([0], [0], color='black', linewidth=1.5, label='Coastlines'),
        Line2D([0], [0], color='red', linewidth=2, label='Subduction Zones'),
        Line2D([0], [0], color='blue', linewidth=2, label='Mid-Ocean Ridges'),
        Line2D([0], [0], color='orange', linewidth=1.5, label='Transform Faults'),
    ]
    ax.legend(handles=legend_elements, loc='lower left', fontsize=9)
    
    # Progress indicator
    if frame % 10 == 0:
        print(f"   Frame {frame}/{n_frames}: {reconstruction_time:.0f} Ma")
    
    return []

# Create animation
anim = FuncAnimation(fig, update, frames=n_frames, interval=1000/FPS, blit=False)

# Save animation
print("\n💾 Saving animation...")
anim.save(OUTPUT_FILE, fps=FPS, dpi=DPI, writer='ffmpeg')
print(f"\n✅ Animation saved as '{OUTPUT_FILE}'")

plt.close()

## 7. Display the Animation

In [ ]:
# Display the video in the notebook
from IPython.display import Video
Video(OUTPUT_FILE, embed=True, width=800)

## 8. Download the Animation

If you're using Google Colab, you can download the animation:

In [ ]:
# Download the file (Google Colab only)
try:
    from google.colab import files
    files.download(OUTPUT_FILE)
    print(f"📥 Downloading {OUTPUT_FILE}...")
except:
    print(f"ℹ️ File saved locally as '{OUTPUT_FILE}'")
    print("   (Download option only available in Google Colab)")

---

## 📚 References

1. Müller, R.D., et al. (2019). A global plate model including lithospheric deformation along major rifts and orogens since the Triassic. *Tectonics*, 38, 1884-1907.

2. Zahirovic, S., et al. (2014). The Cretaceous and Cenozoic tectonic evolution of Southeast Asia. *Solid Earth*, 5, 227-273.

3. Mather, B.R., et al. (2023). Deep time spatio-temporal data analysis using pyGPlates with PlateTectonicTools and GPlately. *Geoscience Data Journal*, 1-8.

---

<center>
    <b>Made with ❤️ by Tin Ko Oo, Mahidol University</b>
</center>